# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

The dataset represents **17 marketing campaigns** conducted by a Portuguese bank between **May 2008 and November 2010**. According to the paper’s *Materials and Methods* section, these campaigns produced a total of **79,354 contacts** in the broader campaign data. :contentReference[oaicite:0]{index=0}

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



## Feature Data Quality Assessment

From examining the dataset and feature descriptions, the following observations can be made:

- There are no explicit missing values (all columns show non-null counts), one contain the value **"unknown"**, which represents missing or incomplete information.
- The **pdays** variable uses the value **999** to indicate that the client was not previously contacted. This is not a true numeric value and should be treated as a special category or transformed.
- Numerical variables (e.g., age, duration, campaign, euribor3m) are already in appropriate numeric formats and do not require type conversion.
- The target variable **y** is categorical ("yes"/"no") and must be converted into a binary variable (e.g., 1/0) for modeling.
- The **duration** variable introduces data leakage, since it is only known after the call is completed. It should be excluded when building realistic predictive models.

Overall, while the dataset does not contain traditional missing values, it requires handling of "unknown" categories, transformation of special values, and proper encoding of categorical variables.

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

## Business Objective

The goal of this project is to help a bank improve the effectiveness of its telemarketing campaigns by identifying which customers are most likely to subscribe to a term deposit.

By understanding the factors that influence customer responses, the bank can:
- Target the right customers
- Reduce unnecessary calls
- Increase conversion rates
- Optimize marketing resources

## Data Mining Objective

From a data science perspective, this is a **binary classification problem**.

The objective is to build predictive models that can classify whether a customer will subscribe to a term deposit (y = "yes" or "no") based on their demographic, financial, and campaign-related attributes.

The models will be evaluated and compared to determine which algorithm performs best in predicting customer responses.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

## Feature Engineering

To prepare the dataset for modeling, we need to:

- Convert the target variable (y) into a binary format (1 = yes, 0 = no)
- Encode categorical variables into numeric format using one-hot encoding
- Remove features that may cause data leakage (e.g., duration)
- Ensure all features are suitable for machine learning models

These steps will allow us to train classification models effectively.

In [5]:
# Convert target variable to binary
df['y'] = df['y'].map({'yes': 1, 'no': 0})

df['y'].value_counts()

y
0    36548
1     4640
Name: count, dtype: int64

In [6]:
# Drop duration to avoid data leakage
df_model = df.drop(columns=['duration'])

# Separate features and target
X = df_model.drop('y', axis=1)
y = df_model['y']

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

X.head()

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,job_blue-collar,...,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success
0,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,True,False,False,False,True,False,False,False,True,False
1,57,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,True,False,False,False,True,False,False,False,True,False
2,37,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,True,False,False,False,True,False,False,False,True,False
3,40,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,True,False,False,False,True,False,False,False,True,False
4,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,True,False,False,False,True,False,False,False,True,False


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

## Train/Test Split

To evaluate model performance, the dataset is split into training and testing sets.

The training set is used to train the models, while the testing set is used to evaluate how well the models generalize to unseen data.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((32950, 52), (8238, 52))

The dataset has been split into training and testing sets using an 80/20 ratio. This ensures that we have sufficient data for training while preserving a portion for unbiased evaluation.

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

## Baseline Model

Before building machine learning models, it is important to establish a baseline performance.

Since the dataset is imbalanced, with the majority of customers not subscribing to the deposit, a simple baseline model would predict the majority class ("no") for all observations.

In [8]:
# Baseline accuracy (predict all as 0 = "no")
baseline_accuracy = (y == 0).mean()

baseline_accuracy

np.float64(0.8873458288821987)

The baseline accuracy is approximately 0.887, meaning that a model that always predicts "no" would be correct about 88.7% of the time.

Therefore, any useful model must perform better than this baseline and, more importantly, should improve the ability to correctly identify customers who will subscribe (the minority class).

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [9]:
from sklearn.linear_model import LogisticRegression

# Initialize model
logreg = LogisticRegression(max_iter=1000)

# Train model
logreg.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

### Problem 9: Score the Model

What is the accuracy of your model?

In [10]:
from sklearn.metrics import accuracy_score

# Predictions
y_pred = logreg.predict(X_test)

# Accuracy
logreg_accuracy = accuracy_score(y_test, y_pred)

logreg_accuracy

0.8969410050983249

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

## Model Comparison

We compare the performance of four classification models:

- Logistic Regression
- K-Nearest Neighbors (KNN)
- Decision Tree
- Support Vector Machine (SVM)

The models are evaluated based on:
- Training time
- Training accuracy
- Test accuracy

In [11]:
import time
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC()
}

results = []

for name, model in models.items():
    start = time.time()
    
    model.fit(X_train, y_train)
    
    end = time.time()
    
    train_time = end - start
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    results.append([name, train_time, train_acc, test_acc])

results_df = pd.DataFrame(results, columns=["Model", "Train Time", "Train Accuracy", "Test Accuracy"])

results_df

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Train Time,Train Accuracy,Test Accuracy
0,Logistic Regression,1.463190,0.901002,0.896941
1,KNN,0.004299,0.914719,0.887473
2,Decision Tree,0.121736,0.995357,0.836732
3,SVM,4.783101,0.898209,0.894756


## Model Comparison Results

The performance of the four models is summarized below:

- **Logistic Regression** achieved the highest test accuracy (~89.7%) and provides a strong balance between performance and interpretability.
- **KNN** performed similarly to the baseline (~88.7%), indicating limited improvement in predictive power.
- **Decision Tree** showed very high training accuracy (~99.5%) but significantly lower test accuracy (~83.9%), indicating strong overfitting.
- **SVM** performed comparably to Logistic Regression (~89.5%) but required significantly longer training time.

### Key Observations

- Logistic Regression and SVM perform best on unseen data.
- Decision Tree overfits the training data and generalizes poorly.
- KNN does not significantly outperform the baseline model.

### Conclusion

Logistic Regression is the most suitable model due to:
- High accuracy
- Good generalization
- Faster training time compared to SVM
- Simpler interpretability

However, due to class imbalance, accuracy alone may not fully reflect model performance, and additional metrics such as precision and recall should be considered.

### Note on Class Imbalance

The dataset is highly imbalanced, with a majority of "no" responses. This means that accuracy alone can be misleading, as a model could achieve high accuracy by predicting only the majority class.

Future improvements should include evaluating models using precision, recall, and F1-score to better assess performance on the minority class ("yes").

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

## Improving the Model

The initial model comparison used default settings and accuracy as the evaluation metric. However, since the dataset is imbalanced, accuracy alone is not sufficient.

For this problem, **recall** is a more appropriate metric because the bank is especially interested in identifying customers who are likely to subscribe to a term deposit.

To improve model performance, we will:
- Tune model hyperparameters using GridSearchCV
- Use recall as the primary evaluation metric
- Compare the improved models to determine the most effective classifier

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score

## Why Recall?

Because the target classes are imbalanced, a model can achieve high accuracy by mostly predicting "no".

Recall helps measure how well the model identifies actual subscribers ("yes"), which is more useful in a marketing campaign setting where missing likely customers can reduce campaign effectiveness.

In [13]:
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

knn_params = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance']
}

knn_grid = GridSearchCV(
    knn_pipe,
    knn_params,
    cv=3,
    scoring='recall',
    n_jobs=-1
)

knn_grid.fit(X_train, y_train)

knn_grid.best_params_, knn_grid.best_score_

({'model__n_neighbors': 3, 'model__weights': 'distance'},
 np.float64(0.29176788124156544))

In [14]:
logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])

logreg_params = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['lbfgs']
}

logreg_grid = GridSearchCV(
    logreg_pipe,
    logreg_params,
    cv=3,
    scoring='recall',
    n_jobs=-1
)

logreg_grid.fit(X_train, y_train)

logreg_grid.best_params_, logreg_grid.best_score_

({'model__C': 1, 'model__solver': 'lbfgs'}, np.float64(0.23265856950067476))

In [15]:
dt_params = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=3,
    scoring='recall',
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

dt_grid.best_params_, dt_grid.best_score_

({'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2},
 np.float64(0.3365721997300945))

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='linear', C=0.1))
])

svm_pipe.fit(X_train, y_train)

y_pred_svm_tuned = svm_pipe.predict(X_test)

print("Recall:", recall_score(y_test, y_pred_svm_tuned))
print("Precision:", precision_score(y_test, y_pred_svm_tuned))
print("F1:", f1_score(y_test, y_pred_svm_tuned))

Recall: 0.20320855614973263
Precision: 0.6089743589743589
F1: 0.30473135525260625


In [17]:
tuned_models = {
    'KNN': knn_grid.best_estimator_,
    'Logistic Regression': logreg_grid.best_estimator_,
    'Decision Tree': dt_grid.best_estimator_,
    'SVM': svm_pipe   # <-- FIX HERE
}

tuned_results = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    tuned_results.append([
        name,
        recall_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ])

tuned_results_df = pd.DataFrame(
    tuned_results,
    columns=['Model', 'Recall', 'Precision', 'F1 Score']
)

tuned_results_df

,Model,Recall,Precision,F1 Score
0,KNN,0.304813,0.405405,0.347985
1,Logistic Regression,0.211765,0.636656,0.317817
2,Decision Tree,0.341176,0.312745,0.326343
3,SVM,0.203209,0.608974,0.304731


In [18]:
best_model_name = tuned_results_df.sort_values(by='Recall', ascending=False).iloc[0]['Model']
best_model = tuned_models[best_model_name]

best_y_pred = best_model.predict(X_test)

print("Best Model:", best_model_name)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_y_pred))

print("\nClassification Report:")
print(classification_report(y_test, best_y_pred))

Best Model: Decision Tree

Confusion Matrix:
[[6602  701]
 [ 616  319]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91      7303
           1       0.31      0.34      0.33       935

    accuracy                           0.84      8238
   macro avg       0.61      0.62      0.62      8238
weighted avg       0.85      0.84      0.84      8238



## Improved Model Findings

After tuning the models and evaluating them using recall, we observe a clear shift in model performance compared to accuracy-based evaluation.

### Model Performance Comparison

- **Decision Tree** achieved the highest recall (~0.34), making it the most effective at identifying customers who are likely to subscribe.
- **KNN** showed moderate recall (~0.30) but lower precision.
- **Logistic Regression** had lower recall (~0.21) but higher precision, indicating it is more conservative in predicting positive outcomes.
- **SVM** also showed low recall (~0.20) but relatively high precision.

### Key Insight

There is a clear tradeoff between **recall and precision** across models:

- Decision Tree → better at capturing potential subscribers (higher recall)
- Logistic Regression & SVM → better at avoiding false positives (higher precision)

This demonstrates that model selection depends heavily on the business objective and chosen evaluation metric.

## Final Recommendation

For this business problem, the **Decision Tree model is the most suitable choice**, as it achieves the highest recall and identifies more potential subscribers.

### Why Decision Tree?

- It captures the largest number of actual "yes" customers
- Missing a potential subscriber is more costly than contacting an uninterested customer
- It aligns well with the goal of improving marketing campaign effectiveness

### Business Impact

Using the Decision Tree model, the bank can:
- Increase the number of successful subscriptions
- Better target customers for telemarketing campaigns
- Improve overall campaign efficiency

### Tradeoff Consideration

The model has lower precision, meaning it may target some customers who are unlikely to subscribe. However, in this context, this is an acceptable tradeoff.

### Future Improvements

- Apply class balancing techniques (e.g., SMOTE)
- Tune decision thresholds
- Explore ensemble models (Random Forest, Gradient Boosting)
- Incorporate additional customer behavior data

Overall, selecting a model based on recall provides better alignment with business goals and leads to more effective decision-making.

##### Questions